<a href="https://colab.research.google.com/github/hriddhisrivastav09/Deep_Learning-Assignments/blob/main/Assignment_04/Assignment_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
# Fix seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# ==========================================
# 1. FETCH REAL HISTORICAL WEATHER DATA
# ==========================================
def fetch_real_weather():
print("Fetching historical weather data from Open-Meteo API...")
url = "https://archive-api.open-meteo.com/v1/archive"
# Coordinates for London, UK (Latitude, Longitude)
params = {
"latitude": 51.5074,
"longitude": -0.1278,
"start_date": "2014-01-01",
"end_date": "2024-01-01",
"daily": "temperature_2m_mean",
"timezone": "Europe/London"
}
response = requests.get(url, params=params)
data = response.json()
df = pd.DataFrame({
"Date": pd.to_datetime(data["daily"]["time"]),
"Temperature": data["daily"]["temperature_2m_mean"]
})
# Clean missing values if any
df["Temperature"] = df["Temperature"].interpolate(method="linear")
print(f"Successfully fetched {len(df)} daily temperature records.")
return df
df = fetch_real_weather()

In [1]:
# ==========================================
# 2. DATA PREPROCESSING & SLIDING WINDOW
# ==========================================
# Scale values strictly to [0, 1] range for stable training
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_temp = scaler.fit_transform(df[["Temperature"]].values)
LOOKBACK = 30 # Look back 30 days to predict the next day's mean temperature
def create_sliding_windows(data, window_size):
X, y = [], []
for i in range(len(data) - window_size):
X.append(data[i : i + window_size])
y.append(data[i + window_size])
return np.array(X), np.array(y)
X, y = create_sliding_windows(scaled_temp, LOOKBACK)
# Chronological Train-Test Split (80% train, 20% test - preserve time dependency)
train_size = int(len(X) * 0.8)
X_train, y_train = X[:train_size], y[:train_size]
X_test, y_test = X[train_size:], y[train_size:]
# Convert to PyTorch Tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)
# PyTorch Dataset and DataLoader
class WeatherDataset(Dataset):
  def init (self, X, y):
    self.X = X
    self.y = y
def len (self):
  return len(self.X)
def getitem (self, idx):
  return self.X[idx], self.y[idx]
train_dataset = WeatherDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

IndentationError: expected an indented block after function definition on line 8 (7177506.py, line 9)

In [ ]:
# ==========================================
# 3. DEFINE LSTM ARCHITECTURE
# ==========================================
class WeatherLSTM(nn.Module):
def init (self, input_size=1, hidden_size=64, num_layers=2, output_size=1):
super(WeatherLSTM, self). init ()
self.lstm = nn.LSTM(
input_size=input_size,
hidden_size=hidden_size,
num_layers=num_layers,
batch_first=True,
dropout=0.2
)
self.fc = nn.Linear(hidden_size, output_size)
def forward(self, x):
# x shape: (batch_size, sequence_length, features)
out, (h_n, c_n) = self.lstm(x)
# Extract output of the last time step in sequence
last_step_out = out[:, -1, :]
prediction = self.fc(last_step_out)
return prediction
model = WeatherLSTM()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# ==========================================
# 4. TRAINING LOOP
# ==========================================
EPOCHS = 40
print("\nStarting Training...")
model.train()
for epoch in range(EPOCHS):
total_loss = 0.0
for batch_X, batch_y in train_loader:
optimizer.zero_grad()
predictions = model(batch_X)
loss = criterion(predictions, batch_y)
loss.backward()
optimizer.step()
total_loss += loss.item() * batch_X.size(0)
epoch_loss = total_loss / len(train_loader.dataset)
if (epoch + 1) % 10 == 0:
print(f"Epoch [{epoch + 1}/{EPOCHS}] - Loss: {epoch_loss:.6f}")

In [ ]:
# ==========================================
# 5. EVALUATION & INVERSE SCALING
# ==========================================
model.eval()
with torch.no_grad():
test_preds_scaled = model(X_test_t).numpy()
# Denormalize predictions back to actual temperature (°C)
test_preds = scaler.inverse_transform(test_preds_scaled)
y_test_actual = scaler.inverse_transform(y_test)
# Calculate Evaluation Metric
rmse = np.sqrt(np.mean((test_preds - y_test_actual) ** 2))
print(f"\nModel Evaluation -> Test RMSE: {rmse:.2f} °C")

In [ ]:
# ==========================================
# 6. VISUALIZATION
# ==========================================
test_dates = df["Date"].iloc[-len(y_test_actual):].values
plt.figure(figsize=(14, 6))
plt.plot(test_dates, y_test_actual, label="Actual Daily Temperature", color="black",
alpha=0.75)
plt.plot(test_dates, test_preds, label="LSTM Predicted Temperature", color="red",
linestyle="--", alpha=0.9)
plt.title("Real Weather Forecasting using LSTM (London, 2022–2024)")
plt.xlabel("Date")
plt.ylabel("Temperature (°C)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()